In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_products
# Source          : employees.csv
# Target          : procurement.silver.silver_products
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned products master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read products data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp
from pyspark.sql.functions import coalesce
from pyspark.sql.functions import expr


In [0]:
# ============================================================
# Read Bronze Product Table
# ============================================================

bronze_products_df = read_delta(BRONZE_PRODUCTS)

preview(bronze_products_df,"Bronze products")

In [0]:
# ============================================================
# Identify invalid products records
# NULL & Blank product id, null product names, null category, null unit_of_measure, invalid price, null currency
# ============================================================
invalid_products = bronze_products_df.filter(

    (col("product_id").isNull() | (trim(col("product_id")) == "")) |

    (col("product_name").isNull() | (trim(col("product_name")) == "")) |

    (col("category").isNull() | (trim(col("category")) == "")) |

    (col("unit_of_measure").isNull() | (trim(col("unit_of_measure")) == "")) |

    (col("standard_unit_price").isNull() | (col("standard_unit_price") <= 0)) |

    (col("currency").isNull() | (trim(col("currency")) == ""))

)   

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_products = (invalid_products
    .withColumn("audit_timestamp", current_timestamp())
    .withColumn("source_table", lit("Products"))
    .withColumn("pipeline_layer", lit("Silver"))
    .withColumn("issue_type", lit("Invalid Record"))
)

display(invalid_products)

In [0]:
# ============================================================
# Write Invalid Product Audit Table
# ============================================================

if invalid_products.count() > 0:
    write_delta(invalid_products,AUDIT_INVALID_PRODUCTS,mode="overwrite")
    print("Invalid products records written.")
else:
    print("No invalid products records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

silver_products_df = (
    bronze_products_df

    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name",initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("unit_of_measure", upper(trim(col("unit_of_measure"))))
    .withColumn("currency", upper(trim(col("currency"))))

    .filter(
        col("product_id").isNotNull()
        & (col("product_id") != "")
        & col("product_name").isNotNull()
        & (col("product_name") != "")
        & col("category").isNotNull()
        & (col("category") != "")
        & col("unit_of_measure").isNotNull()
        & (col("unit_of_measure") != "")
        & col("standard_unit_price").isNotNull()
        & (col("standard_unit_price") > 0)
        & col("currency").isNotNull()
        & (col("currency") != "")
    )
)

display(silver_products_df)

In [0]:
# ============================================================
# Remove Duplicate Product ids
# ============================================================

window_spec = Window.partitionBy("product_id").orderBy("product_id")

silver_products_df = (silver_products_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_products_df,"Silver Employees")


In [0]:
# ============================================================
# Apply Business Transformations
# ============================================================

silver_products_df = (
    silver_products_df

    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("unit_of_measure", upper(trim(col("unit_of_measure"))))
    .withColumn("currency", upper(trim(col("currency"))))
)

display(silver_products_df)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_products_df = (silver_products_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_products_df,table_name=SILVER_PRODUCTS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

print("=" * 60)
print("Silver Employee Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records : {bronze_products_df.count()}")
print(f"Silver Records : {silver_products_df.count()}")
print(f"Duplicates Removed : {bronze_products_df.count() - silver_products_df.count()}")